# CIFAR-100 Training Pipeline

This notebook orchestrates the CIFAR-100 experiment matrix added in `configs/cifar100`.

The pipeline uses 45k training images and 5k validation images from the original CIFAR-100 training split. The official 10k test split is evaluated once on `best.pth` and reported as `test_acc1` and `test_acc5`.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "tools" / "run_cifar100_experiments.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "tools" / "run_cifar100_experiments.py").exists(), f"Run from repo root or notebooks dir. Got {ROOT}"

COC_PYTHON = Path(r"D:\Anaconda\envs\CoC\python.exe")
PYTHON = COC_PYTHON if COC_PYTHON.exists() else Path(sys.executable)

def run(args: list[str]) -> None:
    cmd = [str(PYTHON), *args]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

print("Repo:", ROOT)
print("Python:", PYTHON)

## Phase Switches

Keep `RUN_SMOKE = True` for a first pass. Enable the full matrix only when you are ready for full training.

In [ ]:
RUN_SMOKE = True
RUN_FULL_MATRIX = False
RUN_BENCHMARK_ONLY = False
RUN_SUMMARY_ONLY = False

DATA_ROOT = "data"
RUN_ROOT = "runs_cifar100"
BENCHMARK_ROOT = "results/benchmark_cifar100"
SUMMARY_CSV = "results/cifar100_summary.csv"

SMOKE_EXPERIMENTS = ["resnet18_reference_cifar100", "hbcc_small_no_mix_cifar100"]

## Smoke Check

In [ ]:
if RUN_SMOKE:
    run([
        "tools/run_cifar100_experiments.py",
        "--smoke",
        "--only",
        *SMOKE_EXPERIMENTS,
        "--skip-kd",
        "--data-root",
        DATA_ROOT,
        "--output",
        RUN_ROOT,
        "--benchmark-output",
        BENCHMARK_ROOT,
        "--summary-output",
        SUMMARY_CSV,
    ])

## Full Matrix

In [ ]:
if RUN_FULL_MATRIX:
    run([
        "tools/run_cifar100_experiments.py",
        "--data-root",
        DATA_ROOT,
        "--output",
        RUN_ROOT,
        "--benchmark-output",
        BENCHMARK_ROOT,
        "--summary-output",
        SUMMARY_CSV,
    ])

## Benchmark Or Summary Only

In [ ]:
if RUN_BENCHMARK_ONLY:
    run([
        "tools/run_cifar100_experiments.py",
        "--skip-train",
        "--skip-summary",
        "--data-root",
        DATA_ROOT,
        "--output",
        RUN_ROOT,
        "--benchmark-output",
        BENCHMARK_ROOT,
    ])

if RUN_SUMMARY_ONLY:
    run([
        "tools/cifar100_report.py",
        "--runs",
        RUN_ROOT,
        "--benchmarks",
        BENCHMARK_ROOT,
        "--output",
        SUMMARY_CSV,
    ])

## Review Summary

In [ ]:
summary_path = ROOT / SUMMARY_CSV
if summary_path.exists():
    try:
        import pandas as pd

        display(pd.read_csv(summary_path))
    except ImportError:
        print(summary_path.read_text(encoding="utf-8"))
else:
    print(f"Summary not found yet: {summary_path}")